In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import torch
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import ultralytics

print(ultralytics.__version__)

8.4.104


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_PATH = Path("/content/drive/MyDrive/Data")

ZIP_PATH = DRIVE_PATH / "yolo_dataset.zip"

EXTRACT_PATH = Path("/content/drive/MyDrive/Data")

In [ ]:
import zipfile

EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import yaml

yaml_path = EXTRACT_PATH / "data.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

data["path"] = str(EXTRACT_PATH / "yolo_dataset")

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("Updated data.yaml")

Updated data.yaml


##Loading the model

In [ ]:
model = YOLO("yolov8n.pt")

##Verification of data

In [ ]:
from pathlib import Path

DATASET_PATH = Path("/content/drive/MyDrive/Data/yolo_dataset")

train_images = DATASET_PATH / "train" / "images"
train_labels = DATASET_PATH / "train" / "labels"

valid_images = DATASET_PATH / "valid" / "images"
valid_labels = DATASET_PATH / "valid" / "labels"

test_images = DATASET_PATH / "test" / "images"
test_labels = DATASET_PATH / "test" / "labels"

In [ ]:
def count_files(folder, extension):
    return len(list(folder.glob(f"*{extension}")))

print("Training Images :", count_files(train_images, ".jpg"))
print("Training Labels :", count_files(train_labels, ".txt"))

print()

print("Validation Images :", count_files(valid_images, ".jpg"))
print("Validation Labels :", count_files(valid_labels, ".txt"))

print()

print("Test Images :", count_files(test_images, ".jpg"))
print("Test Labels :", count_files(test_labels, ".txt"))

Training Images : 4500
Training Labels : 4500

Validation Images : 563
Validation Labels : 563

Test Images : 563
Test Labels : 563


In [ ]:
def check_missing_labels(image_folder, label_folder):

    missing = []

    for image in image_folder.glob("*.jpg"):

        label = label_folder / f"{image.stem}.txt"

        if not label.exists():
            missing.append(image.name)

    return missing

In [ ]:
missing_train = check_missing_labels(train_images, train_labels)
missing_valid = check_missing_labels(valid_images, valid_labels)
missing_test = check_missing_labels(test_images, test_labels)

print("Missing train labels :", len(missing_train))
print("Missing valid labels :", len(missing_valid))
print("Missing test labels :", len(missing_test))

Missing train labels : 0
Missing valid labels : 0
Missing test labels : 0


In [ ]:
def find_empty_labels(label_folder):

    empty = []

    for file in label_folder.glob("*.txt"):

        if file.stat().st_size == 0:
            empty.append(file.name)

    return empty

In [ ]:
empty_train = find_empty_labels(train_labels)
empty_valid = find_empty_labels(valid_labels)
empty_test = find_empty_labels(test_labels)

print("Empty train labels :", len(empty_train))
print("Empty valid labels :", len(empty_valid))
print("Empty test labels :", len(empty_test))

Empty train labels : 0
Empty valid labels : 0
Empty test labels : 0


In [ ]:
import yaml

with open("/content/drive/MyDrive/Data/data.yaml", "r") as f:
    config = yaml.safe_load(f)

print(config)

{'path': '/content/drive/MyDrive/Data/yolo_dataset', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: 'articulated_truck', 1: 'bicycle', 2: 'bus', 3: 'car', 4: 'motorcycle', 5: 'motorized_vehicle', 6: 'non-motorized_vehicle', 7: 'pedestrian', 8: 'pickup_truck', 9: 'single_unit_truck', 10: 'work_van'}}


In [ ]:
if (
    len(missing_train) == 0
    and len(missing_valid) == 0
    and len(missing_test) == 0
    and len(empty_train) == 0
    and len(empty_valid) == 0
    and len(empty_test) == 0
):
    print("Dataset is ready for YOLOv8 training!")
else:
    print("Please fix the dataset issues before training.")

Dataset is ready for YOLOv8 training!


##Training

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Data/Training_Results"

results = model.train(
    data="/content/drive/MyDrive/Data/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project=PROJECT_PATH,
    name="YOLOv8n",
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv8n, nbs=64, nms=False, opset=None, opti